# CT-only DPO from existing SFT data (no LLM calls)

This notebook has **three parts**: (1) build CT-style DPO training data; (2) fine-tune on Azure with live event logging; (3) build CT-style DPO *eval* JSONL from validation SFT data and score the deployed model on a stratified dev slice.

**Prerequisite:** `data/generated/sft_data_ct_upload.jsonl` from `SFT/DataGeneration_SFT_v2.ipynb`.

**Outputs (new files only; shared SFT data is never modified):**
- `data/generated/dpo_ct_from_sft_3k.jsonl` — compact rows: `id`, `prompt`, `chosen`, `rejected`, `neg_type`
- `data/generated/dpo_ct_train_upload.jsonl` — Azure DPO rows: `input`, `preferred_output`, `non_preferred_output`

**CT** assistant style matches SFT: `LABEL: While the claim is not … nor …, it is … because …`



---
## Part 1 — Create CT DPO dataset

All steps below use **only local Python** (no teacher model calls).


## Negative strategies (both API-free)

1. **`wrong_label_donor`** — Pick a wrong verdict, then take the **assistant body** (text after `LABEL:`) from a **different** training example whose gold label equals that wrong verdict. The result is CT-shaped but argues the wrong class for *this* claim.

2. **`wrong_label_same_body`** — Keep the **chosen** justification body, but change the leading verdict to a **wrong** label. The sentence often becomes self-contradictory; DPO learns to align the prefix label with the rest of the reasoning (similar in spirit to label–justification mismatch negatives).

Each training example contributes **at most one** rejected example per type; pairs are merged so you get up to **2 × (n/2) == n** pairs when `n` is even (e.g. 3000 → 1500 + 1500). If `n` is odd, one index is left out of both negative pools.


In [ ]:
import json
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

random.seed(42)

CT_SFT_PATH = Path("data/generated/sft_data_ct_upload.jsonl")
# Optional: same CT prompt family on dev — use elsewhere for eval, not used to build DPO pairs here:
CT_VAL_PATH = Path("data/generated/sft_val_ct_upload.jsonl")

PAIRS_OUT = Path("data/generated/dpo_ct_from_sft_3k.jsonl")
UPLOAD_OUT = Path("data/generated/dpo_ct_train_upload.jsonl")

VALID_LABELS = {"SUPPORTED", "CONTRADICTED", "NOT MENTIONED"}
OTHER = {
    "SUPPORTED": ["CONTRADICTED", "NOT MENTIONED"],
    "CONTRADICTED": ["SUPPORTED", "NOT MENTIONED"],
    "NOT MENTIONED": ["SUPPORTED", "CONTRADICTED"],
}


def load_ct_rows(path: Path):
    rows = []
    with path.open(encoding="utf-8") as f:
        for i, line in enumerate(f):
            obj = json.loads(line)
            msgs = obj["messages"]
            system = next(m["content"] for m in msgs if m["role"] == "system")
            user = next(m["content"] for m in msgs if m["role"] == "user")
            assistant = next(m["content"] for m in msgs if m["role"] == "assistant")
            m = re.match(r"^(SUPPORTED|CONTRADICTED|NOT MENTIONED)\s*:\s*(.*)$", assistant.strip(), re.DOTALL)
            if not m:
                raise ValueError(f"line {i}: cannot parse label from assistant: {assistant[:120]!r}")
            label, body = m.group(1), m.group(2).strip()
            if label not in VALID_LABELS:
                raise ValueError(f"line {i}: bad label {label!r}")
            rows.append(
                {
                    "id": i,
                    "system": system,
                    "prompt": user.strip(),
                    "label": label,
                    "assistant_full": assistant.strip(),
                    "body": body,
                }
            )
    return rows


def main():
    if not CT_SFT_PATH.exists():
        raise FileNotFoundError(
            f"Missing {CT_SFT_PATH} — generate it with SFT/DataGeneration_SFT_v2.ipynb first."
        )

    rows = load_ct_rows(CT_SFT_PATH)
    n = len(rows)
    print(f"Loaded {n} CT SFT rows from {CT_SFT_PATH}")
    print("Label counts:", dict(Counter(r["label"] for r in rows)))

    # System prompt must match at DPO time; take from data (identical across rows in practice).
    system_prompt = rows[0]["system"]

    by_label = defaultdict(list)
    for r in rows:
        by_label[r["label"]].append(r)

    order = list(range(n))
    random.shuffle(order)
    half = n // 2
    idx_neg1 = order[:half]
    idx_neg2 = order[half : half * 2]

    chosen_by_id = {r["id"]: r for r in rows}

    pairs = []

    # --- Type 1: wrong label + donor CT body ---
    for i in idx_neg1:
        ex = rows[i]
        wrong = random.choice(OTHER[ex["label"]])
        donor = random.choice(by_label[wrong])
        # Avoid trivial same-row donor when pool allows
        for _ in range(20):
            if donor["id"] != ex["id"]:
                break
            donor = random.choice(by_label[wrong])
        rejected = f"{wrong}: {donor['body']}"
        pairs.append(
            {
                "id": ex["id"],
                "prompt": ex["prompt"],
                "chosen": ex["assistant_full"],
                "rejected": rejected,
                "neg_type": "wrong_label_donor",
            }
        )

    # --- Type 2: wrong label + same chosen body (label / reasoning mismatch) ---
    for i in idx_neg2:
        ex = rows[i]
        wrong = random.choice(OTHER[ex["label"]])
        rejected = f"{wrong}: {ex['body']}"
        pairs.append(
            {
                "id": ex["id"],
                "prompt": ex["prompt"],
                "chosen": ex["assistant_full"],
                "rejected": rejected,
                "neg_type": "wrong_label_same_body",
            }
        )

    random.shuffle(pairs)

    PAIRS_OUT.parent.mkdir(parents=True, exist_ok=True)
    with PAIRS_OUT.open("w", encoding="utf-8") as f:
        for p in pairs:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")
    print(f"Wrote {len(pairs)} pairs → {PAIRS_OUT}")

    def to_upload_row(p):
        return {
            "input": {
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": p["prompt"]},
                ]
            },
            "preferred_output": [{"role": "assistant", "content": p["chosen"]}],
            "non_preferred_output": [{"role": "assistant", "content": p["rejected"]}],
        }

    upload_rows = [to_upload_row(p) for p in pairs]
    with UPLOAD_OUT.open("w", encoding="utf-8") as f:
        for row in upload_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"Wrote {len(upload_rows)} Azure-format rows → {UPLOAD_OUT}")

    print("\nSample compact pair (truncated):")
    print(json.dumps(pairs[0], indent=2, ensure_ascii=False)[:1200])


main()


### Run data build

Run the code cell below. It overwrites only `dpo_ct_from_sft_3k.jsonl` and `dpo_ct_train_upload.jsonl`.


---
## Part 2 — Azure OpenAI DPO fine-tuning


### Credentials, upload, job, and live logs

1. Set **environment variables** (never commit keys): `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_ENDPOINT`, optionally `AZURE_OPENAI_API_VERSION`, `AZURE_DPO_BASE_MODEL`, `AZURE_DPO_SUFFIX`.
2. Run **client** → **upload** → **create job** → **monitor** (prints **every new** fine-tuning event as it appears; step loss is usually inside `message` lines like `Step N: training loss=...`. When available, the **checkpoints** API also prints one line per step with `train_loss`.)

**Base model:** Use the model name your resource accepts (often the **base** SKU string, or a **prior fine-tuned model id** if your workspace requires chaining). Match what you used in `FEVER_DPO2.ipynb`.

**Validation:** `sft_val_ct_upload.jsonl` is plain SFT JSONL. This notebook does not upload it; add `validation_file=...` only if you build a preference-format val file.


In [ ]:
import os
import time
import json
from pathlib import Path
from datetime import datetime, timezone

from openai import AzureOpenAI

# --- Edit or set via environment ---
BASE_MODEL = os.environ.get("AZURE_DPO_BASE_MODEL", "gpt-4.1-nano-2025-04-14")
TRAIN_JSONL = Path("data/generated/dpo_ct_train_upload.jsonl")
SUFFIX = os.environ.get("AZURE_DPO_SUFFIX")

RESOURCE_GROUP = "cis-5270-team-10"
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

OPENAI_ENDPOINT = f"https://{RESOURCE_GROUP}.openai.azure.com"
SUBSCRIPTION_ID = os.environ.get("SUBSCRIPTION_ID")

if not OPENAI_API_KEY or not OPENAI_ENDPOINT:
    raise RuntimeError(
        "Set AZURE_OPENAI_API_KEY and AZURE_OPENAI_ENDPOINT before running the next cells. "
        "Optional: AZURE_DPO_BASE_MODEL, AZURE_DPO_SUFFIX, AZURE_OPENAI_API_VERSION."
    )

openai_client = AzureOpenAI(
    api_key=OPENAI_API_KEY,
    azure_endpoint=OPENAI_ENDPOINT,
    api_version="2025-04-01-preview",
)
# print("AzureOpenAI ready | api_version =", API_VERSION, "| base model =", BASE_MODEL)


In [3]:
assert TRAIN_JSONL.exists(), f"Missing {TRAIN_JSONL} — run the data generation cell first."

print("Uploading DPO training file...")
with TRAIN_JSONL.open("rb") as f:
    train_file = openai_client.files.create(file=f, purpose="fine-tune")

train_file_id = train_file.id
print("Training file ID:", train_file_id)


Uploading DPO training file...
Training file ID: file-25c92a6155f64bb9b95130b0402964ea


In [4]:
print(f"Creating DPO job | model={BASE_MODEL!r} | suffix={SUFFIX!r}")

job = openai_client.fine_tuning.jobs.create(
    training_file=train_file_id,
    model=BASE_MODEL,
    method={
        "type": "dpo",
        "dpo": {
            "hyperparameters": {
                "n_epochs": 1,
                "batch_size": 1,
                "learning_rate_multiplier": 1.0,
            }
        },
    },
    extra_body={"trainingType": "GlobalStandard"},
    suffix=SUFFIX,
)

dpo_job_id = job.id
print("Job ID:", dpo_job_id)
print("Initial status:", job.status)


Creating DPO job | model='gpt-4.1-nano-2025-04-14' | suffix='dpo_ct'
Job ID: ftjob-134a3493e6e84a5a9346ca2bb7640545
Initial status: pending


In [5]:
def _fmt_ts(ts):
    if not ts:
        return "?"
    return datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")


def collect_events_chronological(client, job_id):
    """Return all job events oldest → newest (API returns newest-first pages)."""
    chronological = []
    after = None
    while True:
        page = client.fine_tuning.jobs.list_events(job_id, limit=100, after=after)
        batch = list(page)
        if not batch:
            break
        chronological.extend(reversed(batch))
        after = batch[-1].id
        if len(batch) < 100:
            break
    return chronological


def print_event(ev):
    typ = getattr(ev, "type", None) or ""
    data = getattr(ev, "data", None)
    extra = ""
    if data is not None:
        try:
            extra = " | " + json.dumps(data, default=str)
        except TypeError:
            extra = " | " + repr(data)
    print(f"{_fmt_ts(ev.created_at)} [{ev.level}] ({typ}) {ev.message}{extra}")


def log_checkpoint_metrics(client, job_id, seen_steps):
    """Print per-step metrics when the checkpoints API exists (optional on Azure)."""
    try:
        cks = list(client.fine_tuning.jobs.checkpoints.list(fine_tuning_job_id=job_id))
    except Exception:
        try:
            cks = list(client.fine_tuning.jobs.checkpoints.list(job_id))
        except Exception:
            return seen_steps
    for ck in cks:
        step = getattr(ck, "step_number", None)
        if step is None or step in seen_steps:
            continue
        seen_steps.add(step)
        m = getattr(ck, "metrics", None)
        parts = [f"step={step}"]
        if m is not None:
            for name in (
                "train_loss",
                "train_mean_token_accuracy",
                "valid_loss",
                "valid_mean_token_accuracy",
            ):
                v = getattr(m, name, None)
                if v is not None:
                    parts.append(f"{name}={v}")
        print("  [checkpoint]", " ".join(parts))
    return seen_steps


def monitor_job(client, job_id, poll_seconds=15.0, log_path="data/results/dpo_ct_ft_events.jsonl"):
    Path("data/results").mkdir(parents=True, exist_ok=True)
    seen_event_ids = set()
    seen_ckpt_steps = set()

    print(
        f"Streaming new events for job {job_id!r} (poll {poll_seconds}s). "
        "Stop the cell anytime; the remote job keeps running.\n"
    )

    while True:
        job = client.fine_tuning.jobs.retrieve(job_id)
        events = collect_events_chronological(client, job_id)
        new_n = 0
        for ev in events:
            eid = getattr(ev, "id", None)
            if not eid or eid in seen_event_ids:
                continue
            seen_event_ids.add(eid)
            new_n += 1
            print_event(ev)
            if log_path:
                row = {
                    "job_id": job_id,
                    "id": eid,
                    "created_at": ev.created_at,
                    "level": ev.level,
                    "type": getattr(ev, "type", None),
                    "message": ev.message,
                    "data": getattr(ev, "data", None),
                }
                with open(log_path, "a", encoding="utf-8") as lf:
                    lf.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")

        seen_ckpt_steps = log_checkpoint_metrics(client, job_id, seen_ckpt_steps)

        if new_n == 0:
            tail = events[-1].message if events else "(no events yet)"
            print(f"[status={job.status}] no new events | tail: {tail[:140]}")

        if job.status in {"succeeded", "failed", "cancelled"}:
            print("\n--- Job finished ---")
            print("status:", job.status)
            print("fine_tuned_model:", getattr(job, "fine_tuned_model", None))
            print("error:", getattr(job, "error", None))
            rf = getattr(job, "result_files", None)
            if rf:
                print("result_files:", rf)
            break

        time.sleep(poll_seconds)

    print(f"\nTotal unique events printed: {len(seen_event_ids)}")
    return client.fine_tuning.jobs.retrieve(job_id)


final_job = monitor_job(openai_client, dpo_job_id)
final_job


Streaming new events for job 'ftjob-134a3493e6e84a5a9346ca2bb7640545' (poll 15.0s). Stop the cell anytime; the remote job keeps running.

2026-04-30 01:31:47 UTC [info] (message) Job enqueued. Waiting for jobs ahead to complete.
2026-04-30 01:31:47 UTC [info] (message) Preprocessing running for file training file.
[status=pending] no new events | tail: Preprocessing running for file training file.
[status=pending] no new events | tail: Preprocessing running for file training file.
[status=pending] no new events | tail: Preprocessing running for file training file.
[status=pending] no new events | tail: Preprocessing running for file training file.
[status=pending] no new events | tail: Preprocessing running for file training file.
[status=pending] no new events | tail: Preprocessing running for file training file.
[status=pending] no new events | tail: Preprocessing running for file training file.
[status=pending] no new events | tail: Preprocessing running for file training file.
[sta

APIConnectionError: Connection error.

### Resume monitoring only

If the kernel restarted, set `dpo_job_id = "ftjob-..."` and run the cell below (skip upload / create).


In [6]:
# Re-run the client cell first if the kernel restarted.
dpo_job_id = "ftjob-134a3493e6e84a5a9346ca2bb7640545"
final_job = monitor_job(openai_client, dpo_job_id, poll_seconds=30)
print("Uncomment, set dpo_job_id from the Azure portal or email, then run monitor_job.")


Streaming new events for job 'ftjob-134a3493e6e84a5a9346ca2bb7640545' (poll 30s). Stop the cell anytime; the remote job keeps running.

2026-04-30 01:31:47 UTC [info] (message) Job enqueued. Waiting for jobs ahead to complete.
2026-04-30 01:31:47 UTC [info] (message) Preprocessing running for file training file.
2026-04-30 01:37:31 UTC [info] (message) Preprocessing completed for file training file.
2026-04-30 01:37:45 UTC [info] (message) Job started.
2026-04-30 01:38:02 UTC [info] (message) Data Import started.
2026-04-30 01:38:02 UTC [info] (message) Finetuning started.
2026-04-30 01:42:55 UTC [info] (message) Job queued, waiting for GPUs.
2026-04-30 01:55:42 UTC [info] (message) Training started.
2026-04-30 02:03:46 UTC [info] (metrics) Step 1: training loss=0.6753595471382141 | {"step": 1, "train_loss": 0.6753595471382141, "train_error_rate": 0}
2026-04-30 02:04:05 UTC [info] (metrics) Step 2: training loss=0.6732476949691772 | {"step": 2, "train_loss": 0.6732476949691772, "train_

---
## Part 3 — Evaluation data + dev metrics

**Two artifacts (mirrors Part 1 vs `FEVER_DPO2.ipynb`):**

1. **DPO-format eval file (teacher CT exists):** `SFT/DataGeneration_SFT_v2.ipynb` already produced `data/generated/sft_val_ct_upload.jsonl`. The cell below rebuilds the same *preference JSONL* style as `dpo_ct_train_upload.jsonl` into **`data/generated/dpo_ct_eval_upload.jsonl`** (and a compact twin). No extra LLM calls — same two programmatic negatives as training.

2. **Deployed-model accuracy on held-out passages:** Following `FEVER_DPO2.ipynb`, we stratify examples from **`data/joined/fever_dev_joined.jsonl`**. That notebook takes up to **667 per label** (2001 total); here the default is **`EVAL_N = 1000`** (balanced 334 / 333 / 333). Set environment variable `CT_EVAL_N=2001` if you want the same total count as the 667-per-class FEVER slice.

Metrics match `FEVER_DPO2.ipynb`: overall accuracy, macro accuracy, per-class accuracy, confusion matrix. Rows with unparseable model output are counted under **`UNPARSEABLE`** so row totals match per-class totals.


In [7]:
# --- Build DPO-format eval JSONL from CT validation SFT upload (same logic as Part 1) ---
import json
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

random.seed(43)

CT_VAL_UPLOAD = Path("data/generated/sft_val_ct_upload.jsonl")
EVAL_PAIRS_OUT = Path("data/generated/dpo_ct_eval_from_val.jsonl")
EVAL_UPLOAD_OUT = Path("data/generated/dpo_ct_eval_upload.jsonl")

VALID_LABELS = {"SUPPORTED", "CONTRADICTED", "NOT MENTIONED"}
OTHER = {
    "SUPPORTED": ["CONTRADICTED", "NOT MENTIONED"],
    "CONTRADICTED": ["SUPPORTED", "NOT MENTIONED"],
    "NOT MENTIONED": ["SUPPORTED", "CONTRADICTED"],
}


def load_ct_rows_from_upload(path: Path):
    rows = []
    with path.open(encoding="utf-8") as f:
        for i, line in enumerate(f):
            obj = json.loads(line)
            msgs = obj["messages"]
            system = next(m["content"] for m in msgs if m["role"] == "system")
            user = next(m["content"] for m in msgs if m["role"] == "user")
            assistant = next(m["content"] for m in msgs if m["role"] == "assistant")
            m = re.match(r"^(SUPPORTED|CONTRADICTED|NOT MENTIONED)\s*:\s*(.*)$", assistant.strip(), re.DOTALL)
            if not m:
                raise ValueError(f"{path} line {i}: bad assistant: {assistant[:120]!r}")
            label, body = m.group(1), m.group(2).strip()
            rows.append(
                {
                    "id": i,
                    "system": system,
                    "prompt": user.strip(),
                    "label": label,
                    "assistant_full": assistant.strip(),
                    "body": body,
                }
            )
    return rows


def build_dpo_pairs(rows):
    n = len(rows)
    by_label = defaultdict(list)
    for r in rows:
        by_label[r["label"]].append(r)
    order = list(range(n))
    random.shuffle(order)
    half = n // 2
    idx_neg1, idx_neg2 = order[:half], order[half : half * 2]
    pairs = []
    for i in idx_neg1:
        ex = rows[i]
        wrong = random.choice(OTHER[ex["label"]])
        donor = random.choice(by_label[wrong])
        for _ in range(20):
            if donor["id"] != ex["id"]:
                break
            donor = random.choice(by_label[wrong])
        pairs.append(
            {
                "id": ex["id"],
                "prompt": ex["prompt"],
                "chosen": ex["assistant_full"],
                "rejected": f"{wrong}: {donor['body']}",
                "neg_type": "wrong_label_donor",
            }
        )
    for i in idx_neg2:
        ex = rows[i]
        wrong = random.choice(OTHER[ex["label"]])
        pairs.append(
            {
                "id": ex["id"],
                "prompt": ex["prompt"],
                "chosen": ex["assistant_full"],
                "rejected": f"{wrong}: {ex['body']}",
                "neg_type": "wrong_label_same_body",
            }
        )
    random.shuffle(pairs)
    return pairs, rows[0]["system"]


if not CT_VAL_UPLOAD.exists():
    raise FileNotFoundError(f"Missing {CT_VAL_UPLOAD} — generate with SFT/DataGeneration_SFT_v2.ipynb")

val_rows = load_ct_rows_from_upload(CT_VAL_UPLOAD)
print(f"Loaded {len(val_rows)} CT val rows from {CT_VAL_UPLOAD}")
print("Label counts:", dict(Counter(r["label"] for r in val_rows)))

pairs, system_prompt = build_dpo_pairs(val_rows)
EVAL_PAIRS_OUT.parent.mkdir(parents=True, exist_ok=True)
with EVAL_PAIRS_OUT.open("w", encoding="utf-8") as f:
    for p in pairs:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
print(f"Wrote {len(pairs)} eval pairs → {EVAL_PAIRS_OUT}")


def to_upload_row(p):
    return {
        "input": {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": p["prompt"]},
            ]
        },
        "preferred_output": [{"role": "assistant", "content": p["chosen"]}],
        "non_preferred_output": [{"role": "assistant", "content": p["rejected"]}],
    }


with EVAL_UPLOAD_OUT.open("w", encoding="utf-8") as f:
    for p in pairs:
        f.write(json.dumps(to_upload_row(p), ensure_ascii=False) + "\n")
print(f"Wrote {len(pairs)} Azure-format eval rows → {EVAL_UPLOAD_OUT}")


Loaded 300 CT val rows from data/generated/sft_val_ct_upload.jsonl
Label counts: {'CONTRADICTED': 100, 'SUPPORTED': 100, 'NOT MENTIONED': 100}
Wrote 300 eval pairs → data/generated/dpo_ct_eval_from_val.jsonl
Wrote 300 Azure-format eval rows → data/generated/dpo_ct_eval_upload.jsonl


### Stratified dev sample + `run_eval` + `compute_metrics`

Requires the **Azure client** from Part 2 (`openai_client`). Set `DPO_CT_DEPLOYMENT` to your deployed fine-tuned model name (or edit `DEPLOYMENT_NAME` below).


In [10]:
import json
import os
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

# --- Match FEVER_DPO2: stratify dev by label, cap per class, shuffle ---
random.seed(42)
DEV_PATH = Path("data/joined/fever_dev_joined.jsonl")
EVAL_N = int(os.environ.get("CT_EVAL_N", "1000"))  # set CT_EVAL_N=2001 for FEVER-sized eval

CT_TRAIN_UPLOAD = Path("data/generated/sft_data_ct_upload.jsonl")


def load_jsonl(path: Path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def stratified_dev_sample(dev_rows, target_n: int):
    buckets = defaultdict(list)
    for ex in dev_rows:
        buckets[ex["label"]].append(ex)
    labels = sorted(buckets.keys())
    base, rem = divmod(target_n, len(labels))
    eval_sample = []
    for j, lbl in enumerate(labels):
        k = base + (1 if j < rem else 0)
        items = buckets[lbl][:]
        random.shuffle(items)
        eval_sample.extend(items[:k])
    random.shuffle(eval_sample)
    return eval_sample


def load_ct_system_prompt() -> str:
    line = CT_TRAIN_UPLOAD.read_text(encoding="utf-8").splitlines()[0]
    obj = json.loads(line)
    return next(m["content"] for m in obj["messages"] if m["role"] == "system")


dev_data = load_jsonl(DEV_PATH)
eval_sample = stratified_dev_sample(dev_data, EVAL_N)
print(f"Dev rows: {len(dev_data)} | eval_sample: {len(eval_sample)} (target {EVAL_N})")
print(Counter(ex["label"] for ex in eval_sample))

CT_EVAL_SYSTEM = load_ct_system_prompt()

VALID_LABELS = {"SUPPORTED", "CONTRADICTED", "NOT MENTIONED"}


def extract_label(text):
    if not isinstance(text, str):
        return None
    for lbl in VALID_LABELS:
        if text.upper().startswith(lbl + ":"):
            return lbl
    if text.strip() in VALID_LABELS:
        return text.strip()
    for lbl in sorted(VALID_LABELS, key=len, reverse=True):
        if re.search(rf"\b{re.escape(lbl)}\b", text.upper()):
            return lbl
    return None


def predict(passage, claim, deployment_name, system_prompt):
    user_msg = f"Passage: {passage}\n\nClaim: {claim}"
    try:
        response = openai_client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_msg},
            ],
            temperature=0.0,
            max_tokens=350,
        )
        raw = response.choices[0].message.content.strip()
        pred_label = extract_label(raw)
        return pred_label, raw
    except Exception as e:
        return None, f"ERROR: {e}"


def run_eval(deployment_name, examples, out_path, system_prompt):
    done_ids = set()
    results = []
    if os.path.exists(out_path):
        with open(out_path, "r", encoding="utf-8") as f:
            for line in f:
                r = json.loads(line)
                done_ids.add(r["id"])
                results.append(r)
        print(f"resuming, {len(done_ids)} already done")

    skipped = 0
    with open(out_path, "a", encoding="utf-8") as out_f:
        for i, ex in enumerate(examples):
            if ex["id"] in done_ids:
                continue
            pred_label, raw = predict(
                ex["passage"], ex["claim"], deployment_name, system_prompt
            )
            if pred_label is None:
                skipped += 1
            record = {
                "id": ex["id"],
                "label": ex["label"],
                "pred_label": pred_label,
                "raw": raw,
            }
            out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
            out_f.flush()
            results.append(record)
            if (i + 1) % 200 == 0:
                print(f"[{i+1}/{len(examples)}] skipped={skipped}")
    print(f"done. skipped={skipped}")
    return results


def compute_metrics(results):
    labels = ["SUPPORTED", "CONTRADICTED", "NOT MENTIONED"]
    extra_pred_bucket = "UNPARSEABLE"
    pred_columns = labels + [extra_pred_bucket]
    per_class = {lbl: {"correct": 0, "total": 0} for lbl in labels}
    confusion = {true: {pred: 0 for pred in pred_columns} for true in labels}
    total_correct = 0
    for r in results:
        gt, pred = r["label"], r["pred_label"]
        per_class[gt]["total"] += 1
        pred_bucket = pred if pred in labels else extra_pred_bucket
        confusion[gt][pred_bucket] += 1
        if pred == gt:
            per_class[gt]["correct"] += 1
            total_correct += 1
    macro_acc = sum(
        per_class[lbl]["correct"] / per_class[lbl]["total"]
        for lbl in labels
        if per_class[lbl]["total"] > 0
    ) / len(labels)
    overall_acc = total_correct / len(results)
    unparseable_count = sum(confusion[t][extra_pred_bucket] for t in labels)
    print(f"  overall accuracy: {overall_acc:.3f}  ({total_correct}/{len(results)})")
    print(f"  macro accuracy:   {macro_acc:.3f}")
    print(f"  unparseable preds: {unparseable_count}")
    print()
    print("  Per-class accuracy:")
    for lbl in labels:
        c = per_class[lbl]
        acc = c["correct"] / c["total"] if c["total"] else 0
        print(f"    {lbl:20s}: {acc:.3f} ({c['correct']}/{c['total']})")
    print()
    print("  Confusion matrix (rows=true, cols=pred):")
    col_w = 14
    header = " " * 22 + "".join(f"{lbl[:col_w]:>{col_w}}" for lbl in pred_columns)
    print(header)
    for true_lbl in labels:
        row = f"  {true_lbl:20s}" + "".join(
            f"{confusion[true_lbl][pred_lbl]:>{col_w}}" for pred_lbl in pred_columns
        )
        print(row)
    return {
        "overall_accuracy": overall_acc,
        "macro_accuracy": macro_acc,
        "unparseable_predictions": unparseable_count,
    }


DEPLOYMENT_NAME = os.environ.get(
    "DPO_CT_DEPLOYMENT",
    "1-nano-2025-04-14-dpo_ct",
)
RESULTS_PATH = Path("data/results/dpo_ct_eval_dev_results.jsonl")
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

print("CT DPO eval deployment:", DEPLOYMENT_NAME)
dpo_ct_results = run_eval(
    deployment_name=DEPLOYMENT_NAME,
    examples=eval_sample,
    out_path=str(RESULTS_PATH),
    system_prompt=CT_EVAL_SYSTEM,
)
print("DPO CT fine-tuned — dev metrics")
dpo_ct_metrics = compute_metrics(dpo_ct_results)


Dev rows: 19891 | eval_sample: 1000 (target 1000)
Counter({'CONTRADICTED': 334, 'SUPPORTED': 333, 'NOT MENTIONED': 333})
CT DPO eval deployment: 1-nano-2025-04-14-dpo_ct
[200/1000] skipped=1
[400/1000] skipped=4
[600/1000] skipped=7
[800/1000] skipped=11
[1000/1000] skipped=13
done. skipped=13
DPO CT fine-tuned — dev metrics
  overall accuracy: 0.881  (881/1000)
  macro accuracy:   0.881
  unparseable preds: 13

  Per-class accuracy:
    SUPPORTED           : 0.970 (323/333)
    CONTRADICTED        : 0.719 (240/334)
    NOT MENTIONED       : 0.955 (318/333)

  Confusion matrix (rows=true, cols=pred):
                           SUPPORTED  CONTRADICTED NOT MENTIONED   UNPARSEABLE
  SUPPORTED                      323             8             2             0
  CONTRADICTED                    51           240            35             8
  NOT MENTIONED                    6             4           318             5


### Data Cleaning Step for the result file
There are cases where the parser failed or the model didn't correctly produced the prediction label although it answered a label in justification. We reparsed them for a more accurate result.

In [18]:
i=0
for d in dpo_ct_results:
    if d['label'] == 'CONTRADICTED' and d['pred_label'] == 'NOT MENTIONED':
        print(d['raw'])
        i+=1
print(i)

CONTRADICT: While the claim is not SUPPORTED because Stomp the passage is a film released through Rainforest Films, and the claim is not NOT MENTIONED because the passage does not mention the passage is a web page, it is CONTRADIED because Stomp the passage is a film and Stomp the passage is not Stomp the Stomp the Yard is an Internet forum, which is CONTRADICT the claim.
CONTRADICT: While the passage is supported by support, Ron Underwood rejected all offers to direct Ron Underwood is CONTRADICT because the passage confirms he was directing Ron Underwood, and Ron Underwood is not NOT MENTIONED as the passage explicitly states he is supporting Ron Underwood, so Ron Underwood is CONTRADICT: he is not NOT MENTIONED, and Ron Underwood is CONTRADICT himself as Ron Underwood is rejecting Ron Underwood's claim to Ron Underwood rejecting Ron Underwood's claim as Ron Underwood himself is rejecting Ron Underwood's claim as Ron Underwood's statement is Contradictory, and Ron Underwood's claim is

Fix pred_label to CONSTRADICTED if the justification starts with it

In [20]:
i=0
for d in dpo_ct_results:
    if d['label'] == 'CONTRADICTED' and d['pred_label'] == 'NOT MENTIONED' and d['raw'][:10] == 'CONTRADICT':
        d['pred_label'] = 'CONTRADICTED'
        i+=1
print(i)

33


In [ ]:
i=0
for d in dpo_ct_results:
    if d['label'] == 'CONTRADICTED' and d['pred_label'] == 'SUPPORTED' and d['raw'][:10] == 'CONTRADICT':
        print(d['raw'])
        d['pred_label'] = 'SUPPORTED'
        i+=1
print(i)

CONTRADICT: While the passage is supported by support of Tylenol as a support system, Tyler is CONTRADICT because Tylenol is not supported as Tylenol as Pain Killer, and Tyler is CONTRADICT because the passage does not support Tylenol as Tylenol as Pain Killer, instead of Tyler's claim is Tyler's claim of Tyler as Tylenol as Tyler's claim of Tyler's Tyler as Tyler's Tyler as Tyler's Tyler as Tyler's Tyler as Tyler's claim as Tyler's Tyler as Tyler's claim as Tyler's claim as Tyler's claim as Tyler's claim as Tyler's claim as Tyler's statement of Tyler's Tyler's claim of Tyler's Tyler's claim as Tyler's Tyler's claim as TyTyler's claim as TyTyler's claim as TyTyler's claim as TyTyler's TyTyler's TyTyler's TyTyler's TyTyler's TyTyler's TyTyler's claim as TyTyler's TyTyler's TyTyler's claim as TyTyler's TyTyler's claim as TyTyler's TyTyler's TyTyler's rule of TyTyler's rule of TyTyler's rule of TyTyler's rule of TyTyler's rule of TyTyler's rule of TyTyler's TyTyler's rule of TyTyler's TyT

In [15]:
labels = ["SUPPORTED", "CONTRADICTED", "NOT MENTIONED"]
extra_pred_bucket = "UNPARSEABLE"
# pred_bucket = pred if pred in labels else extra_pred_bucket
for d in dpo_ct_results:
    # if d['label'] == 'CONTRADICTED' and d['pred_label'] == 'UNPARSEABLE':
    if d['pred_label'] not in labels:
        print(d['raw'])

CONTRADICTIED: The passage states Ron Ron Ron Dennis is Global Ron Ron Ron Ron Dennis is Global Consultant, while Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Rons Instead of Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Rons Just Ron Ron Ron Ron Ron Ron Ron Ron Ron Ron Rons Just Ron Rons Rons Rons Rons Rons Rons Rons Rons Rons Rons Rons Rons Rons Rons Rons Rons Rons Rons Rons Rons Rons Rons R Ron Ron Rons R Ron Rons R R R R R R R R R R R R R R R 

In [32]:
for d in dpo_ct_results:
    if d['pred_label'] not in labels:
        if d['raw'][:6] == 'CONTRA':
            d['pred_label'] = 'CONTRADICTED'
        if d['raw'][:5] == 'NOT M':
            d['pred_label'] = 'NOT MENTIONED'

In [16]:
for d in dpo_ct_results:
    if d['label'] == 'NOT MENTIONED' and d['pred_label'] == 'SUPPORTED':
        print(d['raw'])

NOT Supported: The passage does not mention Chris Bosh as a person or his role as tennis player, nor does it contradict him, so the claim does not support Chris Bosh as a person who is Chris Bosh is a tennis player; the passage does not provide information about Chris Bosh's role, and Chris Bosh is not Chris Bosh as Chris Bosh is not Chris Bosh as the claim is NOTSupported; the claim is NOTSupported because the passage does not confirm Chris Bosh as a player, and Chris Bosh is not Chris Bosh as the passage does not provide information about Chris Bosh as Chris Bosh is not Chris Bosh as the claim is NOTSupported; the claim is NOTSupported as the passage does not confirm Chris Bosh as a person of Chris Bosh is not Chris Bosh as the claim is not supported, so the claim is NOTSupported as the passage does not confirm Chris Bosh as Chris Bosh as the claim is not supported; the claim is not supported by the passage's content. Chris Bosh is not Chris Bosh as the passage does not confirm Chris

In [35]:
for d in dpo_ct_results:
    if d['label'] == 'NOT MENTIONED' and d['pred_label'] == 'SUPPORTED':
        if d['raw'][:3] == 'NOT':
            d['pred_label'] = 'NOT MENTIONED'

In [17]:
for d in dpo_ct_results:
    if d['label'] == 'NOT MENTIONED' and d['pred_label'] == 'CONTRADICTED':
        print(d['raw'])

CONTRADICTED: The passage states Scandal is from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Contradicting Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Contradicting Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from Scandal from 

In [36]:
for d in dpo_ct_results:
    if d['label'] == 'NOT MENTIONED' and d['pred_label'] == 'CONTRADICTED':
        if d['raw'][:3] == 'NOT':
            d['pred_label'] = 'NOT MENTIONED'

### Recalculate the metrics based on the cleaned results

In [37]:
dpo_ct_metrics = compute_metrics(dpo_ct_results)

  overall accuracy: 0.927  (927/1000)
  macro accuracy:   0.927
  unparseable preds: 3

  Per-class accuracy:
    SUPPORTED           : 0.970 (323/333)
    CONTRADICTED        : 0.838 (280/334)
    NOT MENTIONED       : 0.973 (324/333)

  Confusion matrix (rows=true, cols=pred):
                           SUPPORTED  CONTRADICTED NOT MENTIONED   UNPARSEABLE
  SUPPORTED                      323             8             2             0
  CONTRADICTED                    51           280             2             1
  NOT MENTIONED                    3             4           324             2


### Checking Reward Hacking using heuristic: degenerate / repetitive `raw` responses

Very long `raw` strings often indicate token-loop style response. This cell summarizes **character length** of `dpo_ct_results[i]['raw']`, picks a default **length threshold** (tunable), and prints the longest cases so you can judge whether the cutoff is reasonable.

**Other cheap heuristics** (not computed below unless you extend the cell):
- **Compression ratio** `len(gzip.compress(raw.encode())) / len(raw)` — repetitive text compresses to a small fraction of its length.
- **Type–token ratio** `len(set(words)) / len(words)` — collapse approaches 0 when the model stutters.
- **Max run of the same token** after whitespace split — reward hacks often repeat one word dozens of times.
- **Max character run** (same letter/digit repeated) — catches `aaaa…` style loops.


In [ ]:
import json
import os
import statistics
from pathlib import Path

def raw_chars(d):
    r = d.get("raw")
    return len(r) if isinstance(r, str) else 0


lens = [raw_chars(d) for d in dpo_ct_results]
n = len(lens)
mean_len = statistics.mean(lens)
median_len = statistics.median(lens)
stdev_len = statistics.stdev(lens) if n > 1 else 0.0
mx = max(lens)
sl = sorted(lens)


def pct(p):
    if not sl:
        return 0.0
    k = min(len(sl) - 1, max(0, int(round((p / 100.0) * (len(sl) - 1)))))
    return float(sl[k])


p90, p95, p99 = pct(90), pct(95), pct(99)

print(f"n = {n}")
print(f"mean raw length (chars):   {mean_len:.1f}")
print(f"median raw length (chars): {median_len:.1f}")
print(f"stdev (chars):             {stdev_len:.1f}")
print(f"p90 / p95 / p99 (chars):   {p90:.0f} / {p95:.0f} / {p99:.0f}")
print(f"max (chars):               {mx}")


THRESH = max(p95, mean_len + 3.0 * stdev_len)
print(
    f"\nDefault threshold: max(p95, mean+3*stdev) = {THRESH:.0f} chars "
    "(set CT_RAW_LEN_THRESH to override)"
)

over = [(d, raw_chars(d)) for d in dpo_ct_results if raw_chars(d) > THRESH]
over.sort(key=lambda x: -x[1])
print(f"Rows over threshold: {len(over)} ({100.0 * len(over) / n:.2f}%)")



n = 1000
mean raw length (chars):   596.6
median raw length (chars): 314.0
stdev (chars):             600.3
p90 / p95 / p99 (chars):   1598 / 1919 / 2478
max (chars):               3059

Default threshold: max(p95, mean+3*stdev) = 2398 chars (set CT_RAW_LEN_THRESH to override)
Rows over threshold: 17 (1.70%)


In [ ]:
over2 = [(d, raw_chars(d)) for d in dpo_ct_results if raw_chars(d) > 596.6]
len(over2)

245

In [49]:
over2[0:10]

[({'id': 29112,
   'label': 'SUPPORTED',
   'pred_label': 'SUPPORTED',
   'raw': 'Supported: The passage states Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Support: Battle of Battle of Battle of Battle of Battle of Battle of Battle of Battle of Battle of Battle of Battle of Battle of Battle of Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle Battle 